# Projection-based WF-in-DFT embedding

A cheap DFT calculation on the whole molecule, then a correlated wavefunction on a fragment of
it, with the environment kept at the DFT level. Runs restricted closed- or open-shell.

The physics follows Manby *et al.*, [JCTC **8**, 2564 (2012)](https://doi.org/10.1021/ct300544e);
the energy expression is Eq. 8 of Goodpaster *et al.*,
[JCP **140**, 18A507 (2014)](https://doi.org/10.1063/1.4864040).

## Roadmap

1. Split the global DFT energy into fragment, environment and cross terms.
2. Localise, then choose which occupied orbitals belong to the fragment (`nbed.act_env_space`).
3. Build the embedded SCF (`nbed.emb_scf`) with either projector and validate against
   DFT-in-DFT, which must reproduce the global DFT energy *exactly*.
4. Swap the fragment solver for HF, then CASCI.
5. Map the fragment onto a qubit Hamiltonian (`nbed.hamiltonian`) and confirm it reproduces
   CASCI.

## Three ways to project, and why they are not interchangeable

All three replace the environment's occupied orbitals with something the fragment's aufbau
filling cannot reach, but they do it differently:

| `proj_type` | environment placed at | exact? | where the environment lands |
|---|---|---|---|
| `"mu"` | $+\mu$, a number you pick | no, error $\mathcal{O}(1/\mu)$ | always the last columns |
| `"huz"` | $-\varepsilon_\text{env}$, set by the system | yes, to machine precision | scattered among the active virtuals |
| `"huz"` + `huz_level_shift=`$\lambda$ | $\lambda - \varepsilon_\text{env}$ | yes, same energy as `"huz"` | always the last columns, for $\lambda$ large enough |

The third row is the one to reach for by default. Huzinaga is exact because it annihilates the
fragment–environment coupling block outright, and a constant $\lambda$ added on top cannot undo
that, so it buys ordering headroom for free. It matters when an environment orbital has a
*positive* eigenvalue: the Huzinaga sign flip then moves it **down**, and if it drops below the
fragment HOMO the embedding silently breaks. Section 5.1.1 explains the mechanism, and
`2-huz_shift.ipynb` shows a system where plain `"huz"` fails and $\lambda$ repairs it.

## Three things that silently give wrong answers

Each is demonstrated numerically below rather than just asserted:

- the projector must span the **occupied** environment orbitals only;
- the embedding correction contracts $v_\text{emb}$ with the **DFT** fragment density, not with
  the solver's density;
- **never select orbitals by column index after embedding.** The embedded SCF returns its own
  orbitals sorted by energy. With Huzinaga the environment is interleaved with the active
  virtuals, so a contiguous CAS window silently swallows environment orbitals. Select by
  projection weight onto the environment space instead; `build_emb_dft` and `build_emb_hf`
  return `env_cols` for exactly this purpose.

In [1]:
import numpy as np
from pyscf import cc, ci, dft, gto, lo, mcscf, scf

from nbed.act_env_space import lowdin_populations, orbital_spread
from nbed.emb_scf import EmbedSCF

np.set_printoptions(linewidth=110, suppress=True)

## 1. Splitting the DFT energy in two

Run KS-DFT on the whole molecule and split its occupied orbitals into two disjoint sets, a
fragment $A$ and an environment $B$. Because the sets are disjoint and the orbitals are
orthonormal, the density splits exactly:

$$\gamma = \gamma_A + \gamma_B .$$

The *energy* does not split exactly, because it is not linear in the density. Define the
leftover as the non-additive energy,

$$E_\text{nad}[\gamma_A,\gamma_B] \equiv E_\text{DFT}[\gamma_A+\gamma_B] - E_\text{DFT}[\gamma_A] - E_\text{DFT}[\gamma_B],$$

so that

$$E_\text{DFT}[\gamma] = E_\text{DFT}[\gamma_A] + E_\text{DFT}[\gamma_B] + E_\text{nad}[\gamma_A,\gamma_B].$$

Nothing is approximated yet: $E_\text{nad}$ is *defined* by this equation. It collects the
inter-subsystem Coulomb repulsion and the non-additive exchange-correlation energy.

The reason to build the partition from *orbitals* rather than from densities alone is that there
is **no non-additive kinetic energy term**. Both subsystems are described by orthonormal orbital
sets drawn from one Slater determinant, so the kinetic energy is additive by construction.
Approximating the non-additive kinetic energy is the dominant error in pure density-based
(frozen-density) embedding, and projection-based embedding sidesteps it entirely. That is what
makes DFT-in-DFT exact here, which section 5 verifies numerically.

`EmbedSCF` computes these on construction, using electronic energies only (`energy_elec`, which
excludes nuclear repulsion) and evaluating each subsystem density in the field of *all* nuclei,
so $E_\text{nuc}$ enters the total exactly once:

| attribute | meaning |
|---|---|
| `E_act` | $E_\text{DFT}[\gamma_A]$ |
| `E_env` | $E_\text{DFT}[\gamma_B]$ |
| `E_cross` | $E_\text{nad}[\gamma_A,\gamma_B]$ |
| `G_emb_ao` | the two-electron part of $v_\text{emb}$, $G[\gamma] - G[\gamma_A]$ |

## 2. The system

Methanol, with the hydroxyl group as the fragment. `6-31G` and `b3lyp` keep every step to a
second or two, so the whole notebook can be re-run while changing the partition.

Set `spin = 0` for the **closed-shell** ground state or `spin = 2` for the **open-shell** triplet; the notebook
picks `RKS`/`RHF` or `ROKS`/`ROHF` accordingly.

In [2]:
geometry = [
    ("O", (-0.6582, -0.0067,  0.1730)),
    ("H", (-1.1326, -0.0311, -0.6482)),
    ("C", ( 0.7031,  0.0083, -0.1305)),
    ("H", ( 0.9877,  0.8943, -0.7114)),
    ("H", ( 1.0155, -0.8918, -0.6742)),
    ("H", ( 1.2001,  0.0363,  0.8431)),
]

basis_set  = "6-31G"
xc         = "b3lyp"
charge     = 0
spin       = 2          # 2S; try 0 as well
max_memory = 10_000     # MB

## the fragment: hydroxyl O and H, as 0-based atom indices
active_atm_idx = [0, 1]
n_occ_active   = 3      # occupied orbitals given to the fragment -> its electron count
n_vir_active   = 3      # extra virtuals in the fragment block
max_spread     = 2.0    # reject orbitals more diffuse than this, in Bohr
mu_val         = 1e9    # level shift for the mu projector

mol = gto.Mole(atom=geometry, basis=basis_set, charge=charge, spin=spin,
               max_memory=max_memory).build()

open_shell = mol.spin != 0
print(f"nao = {mol.nao}   nelec = {mol.nelec}   open_shell = {open_shell}")

nao = 26   nelec = (10, 8)   open_shell = True


In [3]:
## the cheap calculation on the whole molecule: everything downstream is built from this
global_scf = dft.ROKS(mol, xc=xc) if open_shell else dft.RKS(mol, xc=xc)
global_scf.max_cycles = 300

global_scf.kernel()
Sao = global_scf.get_ovlp()

assert global_scf.converged, "global DFT did not converge"
print(f"\nglobal {xc.upper()} = {global_scf.e_tot:.10f}")

<class 'pyscf.dft.roks.ROKS'> does not have attributes  max_cycles


converged SCF energy = -115.403719541911

global B3LYP = -115.4037195419


In [4]:
## whole-molecule wavefunction references, for context later on. The correlated methods must
## match the spin state: an ROHF reference needs the U-variants, otherwise PySCF silently
## converts to UHF and warns.
mf_hf = (scf.ROHF(mol) if open_shell else scf.RHF(mol)).run()
ci_full = (ci.UCISD(mf_hf) if open_shell else ci.CISD(mf_hf)).run()
cc_full = (cc.UCCSD(mf_hf) if open_shell else cc.CCSD(mf_hf)).run()

print(f"\nwhole molecule:  {xc.upper()} {global_scf.e_tot:.8f}   HF {mf_hf.e_tot:.8f}   "
      f"CISD {ci_full.e_tot:.8f}   CCSD {cc_full.e_tot:.8f}")

converged SCF energy = -114.720928305621
E(UCISD) = -114.922449556955  E_corr = -0.2015212513337987
E(UCCSD) = -114.9379881253072  E_corr = -0.2170598196860483

whole molecule:  B3LYP -115.40371954   HF -114.72092831   CISD -114.92244956   CCSD -114.93798813


## 3. Localise first

Canonical KS orbitals are delocalised, so no single one of them is "the O–H bond": the split
would be fuzzy and the fragment would end up sharing orbitals with the environment. A unitary
rotation *within* an occupation block leaves $\gamma$, and therefore the global DFT energy,
completely unchanged, but it makes the orbitals atom-centred and the partition clean. That is
free accuracy, so do it.

Rotate within each block **separately** — doubly occupied, singly occupied, virtual — so that
`mo_occ` still describes column $i$ afterwards. Mixing a virtual into an occupied would change
the density and break everything downstream, which is what the assert guards against.

Localising the virtual block matters more than it looks: canonical virtuals are diffuse, so
under a `max_spread` cut almost none of them are eligible and the fragment ends up with no
virtual orbitals at all.

In [5]:
def localise_blocks(mol, mf, verbose=True):
    """Pipek-Mezey within each occupation block, leaving the total density untouched.

    A block with fewer than two orbitals has nothing to mix, so it is passed through.
    """
    C = mf.mo_coeff.copy()
    blocks = {
        "doubly occupied": mf.mo_occ > 1,
        "singly occupied": mf.mo_occ == 1,
        "virtual":         mf.mo_occ == 0,
    }
    for name, mask in blocks.items():
        if mask.sum() > 1:
            C[:, mask] = lo.PipekMezey(mol, mf.mo_coeff[:, mask]).kernel()
        if verbose:
            print(f"  {name:>16}: {mask.sum():>2} orbitals")

    assert np.allclose(
        mf.make_rdm1(mo_coeff=C, mo_occ=mf.mo_occ),
        mf.make_rdm1(mo_coeff=mf.mo_coeff, mo_occ=mf.mo_occ),
        atol=1e-9,
    ), "localisation changed the density"
    return C


C_loc = localise_blocks(mol, global_scf)

   doubly occupied:  8 orbitals
   singly occupied:  2 orbitals
           virtual: 16 orbitals


## 4. Which orbitals belong to the fragment

`lowdin_populations` scores every MO by how much of it sits on the target atoms. Orthogonalise
the AO basis with $S^{1/2}$, so that squared coefficients genuinely partition an orbital, then
sum them over the target atoms' AOs:

$$w_i = \sum_{\mu \in A} \left[ (S^{1/2} C)_{\mu i} \right]^2 \in [0,1].$$

Because the basis is orthogonalised first, $\sum_\mu$ over *all* atoms gives exactly 1 for every
orbital, so $w_i$ reads directly as "the fraction of orbital $i$ living on the fragment". A raw
Mulliken sum over the same AOs is not bounded this way and can even go negative. Core $1s$ AOs
are dropped beyond helium so tight cores cannot dominate the score; H keeps its $1s$, which is
its valence shell.

Occupied and virtual orbitals are ranked separately, and no MO-index window is used: a strongly
fragment-localised orbital high in the virtual space is usually a compact $\sigma^*$, exactly
what bond breaking needs. `orbital_spread` (the RMS extent
$\sqrt{\langle r^2\rangle - \langle r\rangle^2}$) is the honest way to reject genuinely diffuse
Rydberg-like orbitals instead.

**Only the occupied partition affects the embedding.** The fragment occupied orbitals fix the
subsystem electron count and the environment occupied orbitals build the projector. The selected
*virtuals* never enter the embedded SCF, because after embedding the subsystem has its own
virtual space; they only influence how large a CAS is available to a post-embedding solver.

In [6]:
def pick_fragment(mol, mf, C, atom_idx, n_occ, n_vir, max_spread=None):
    """Rank occupied and virtual orbitals separately by target-atom population."""
    _, population = lowdin_populations(mol, C, atom_idx, drop_core_1s=True)
    spread = orbital_spread(mol, C)

    eligible = np.ones_like(population, dtype=bool)
    if max_spread is not None:
        eligible &= spread <= max_spread

    occ_pool = np.where((mf.mo_occ > 0) & eligible)[0]
    vir_pool = np.where((mf.mo_occ == 0) & eligible)[0]
    if len(occ_pool) < n_occ or len(vir_pool) < n_vir:
        raise ValueError(
            f"asked for {n_occ} occupied and {n_vir} virtual orbitals but only "
            f"{len(occ_pool)} and {len(vir_pool)} are eligible; relax max_spread "
            "or shrink the fragment"
        )

    pick_occ = np.sort(occ_pool[np.argsort(-population[occ_pool])[:n_occ]])
    pick_vir = np.sort(vir_pool[np.argsort(-population[vir_pool])[:n_vir]])
    active_idxs = np.concatenate([pick_occ, pick_vir])
    env_idxs = np.setdiff1d(np.arange(mol.nao), active_idxs)
    return active_idxs, env_idxs, population, spread


active_idxs, env_idxs, population, spread = pick_fragment(
    mol, global_scf, C_loc, active_atm_idx, n_occ_active, n_vir_active, max_spread
)

print(f"fragment MOs {active_idxs}")
print(f"  occupancy   {global_scf.mo_occ[active_idxs]}")
print(f"  OH weight   {np.round(population[active_idxs], 3)}")
print(f"  spread      {np.round(spread[active_idxs], 2)}")
print(f"\nenvironment: {len(env_idxs)} orbitals, "
      f"{int((global_scf.mo_occ[env_idxs] > 0).sum())} of them occupied")
assert len(active_idxs) + len(env_idxs) == mol.nao

fragment MOs [ 2  3  6 12 17 19]
  occupancy   [2. 2. 2. 0. 0. 0.]
  OH weight   [0.65  0.979 0.965 0.035 0.981 0.985]
  spread      [1.35 1.23 1.26 1.8  1.96 1.94]

environment: 20 orbitals, 7 of them occupied


## 5. The embedded SCF

`EmbedSCF` reorders the orbitals into PySCF's CAS layout,

$$C_\text{reidx} = [\;\underbrace{\text{env occ}}_{n_\text{core}}\;|\;\underbrace{\text{frag occ}\;|\;\text{frag vir}}_{n_\text{act}}\;|\;\text{env vir}\;]$$

builds $\gamma$, $\gamma_A$, $\gamma_B$ from those coefficients, asserts they are additive, and
sets up a subsystem `Mole` carrying only the fragment's electrons. Two attributes matter most:

- `env_idx_occ` — the occupied environment columns, which build the projector;
- `ncore` — how many orbitals get shifted, and so how many a correlated solver must discard.

Note that the environment *virtuals* stay in the subsystem's variational space. They are
unoccupied in both subsystems, so nothing needs projecting out, and keeping them gives the
fragment a complete basis to relax in.

In [7]:
emb_obj = EmbedSCF(
    global_scf,
    active_idxs,
    env_idxs,
    C_loc,
    global_scf.mo_occ,
    Sao,
    max_memory,
    mu_val=mu_val,
)

n_occ_total = int((global_scf.mo_occ > 0).sum())
print(f"SCF type            : {emb_obj.SCF_type}")
print(f"subsystem nelec     : {emb_obj.mol_act.nelec}")
print(f"env occupied columns: {emb_obj.env_idx_occ}  (ncore = {emb_obj.ncore})")
print(f"fragment columns    : {emb_obj.act_cols}")
print(f"\nE_act   = {emb_obj.E_act:.8f}")
print(f"E_env   = {emb_obj.E_env:.8f}")
print(f"E_cross = {emb_obj.E_cross:.8f}")
print(f"sum     = {emb_obj.E_act + emb_obj.E_env + emb_obj.E_cross:.8f}"
      f"   (E_DFT_global = {emb_obj.E_DFT_global:.8f})")

## the partition must be the one we asked for: n_occ_active orbitals in, the rest projected out
assert len(emb_obj.env_idx_occ) == n_occ_total - n_occ_active
assert np.isclose(emb_obj.E_act + emb_obj.E_env + emb_obj.E_cross, emb_obj.E_DFT_global)

SCF type            : open-shell
subsystem nelec     : (3, 3)
env occupied columns: [0 1 2 3 4 5 6]  (ncore = 7)
fragment columns    : [ 7  8  9 10 11 12]

E_act   = -50.29342889
E_env   = -140.42582933
E_cross = 34.52501177
sum     = -156.19424645   (E_DFT_global = -156.19424645)


### 5.1 DFT-in-DFT: the test that has to be exact

Use the *same* functional inside and outside. The fragment then has no better description than
the environment, so the embedding cannot buy anything, and

$$E_\text{DFT-in-DFT} = E_\text{DFT}[\gamma]$$

must hold to machine precision. Any deviation is a bug in the projector, in the embedding
potential, or in the correction term — not physics. This is the single most useful check in the
whole notebook.

The total is assembled as

$$E = \underbrace{E_\text{DFT}[\gamma_A^\text{emb}]}_{\texttt{e\_tot}} + \underbrace{E_\text{env} + E_\text{cross} - \mathrm{tr}[\gamma_A\, v_\text{emb}]}_{\texttt{env\_plus\_corrections}}$$

where the subtracted trace uses the **DFT** fragment density $\gamma_A$, not the converged
embedded one. It removes the embedding potential's contribution that `e_tot` already counted, so
it must be evaluated with the density the potential was built from.

#### The $\mu$-shift is only as good as $\mu$

In the $\mu$ shift approach the following term is added to the core Hamiltonian:

$$P =\mu\, S\gamma_B S $$

This pushes the environment orbitals to $+\mu$, but they are only
orthogonal to the fragment in the $\mu \to \infty$ limit.Both the energy error and the residual
overlap fall off as $1/\mu$, exactly as the table below shows, so $\mu$ trades one arbitrary
constant against accuracy. Around $\mu \approx 10^{7}$ the $1/\mu$ error drops below the SCF
convergence floor and the numbers stop improving.

Why does it have this form? The reason is easiest to explain in terms of the MOs basis rather than AO basis...

$$ P_{\mu}^{ao} =\mu\, S\gamma_B S =\mu\, S C_{B}^{occ} C_{B}^{occ,\dagger} S $$

Going to MO basis we do:

$$ P_{\mu}^{mo} =\mu\, C^{\dagger} S C_{B}^{occ} C_{B}^{occ,\dagger} S C$$

$$ P_{\mu}^{mo} =\mu\, I_{occ}^{B} I_{occ}^{B} = \mu I_{occ}^{B}$$

Aka this is just a diagonal matrix with $+\mu$ values in diagonal entries
for every occupied environment index (otherwise zero everywhere else). This
essentially penalizes the energy of these orbitals by $\mu$  






In [8]:
def env_overlap(emb_obj, mf, Sao):
    """Largest overlap between an occupied environment orbital and an occupied embedded one.

    This is the quantity WF-in-DFT needs to vanish: the fragment must not re-occupy anything
    the environment already holds.
    """
    C_env = emb_obj.C_full_reidx[:, emb_obj.env_idx_occ]
    return np.abs(C_env.conj().T @ Sao @ mf.mo_coeff[:, mf.mo_occ > 0]).max()


print(f"{'mu':>9}  {'E error / Ha':>14}  {'max overlap':>12}")
for mu in (1e3, 1e5, 1e7, 1e9):
    emb_obj.mu_val = mu
    e_mu, mf_mu, *_ = emb_obj.build_emb_dft(xc, proj_type="mu")
    print(f"{mu:>9.0e}  {e_mu - global_scf.e_tot:>+14.3e}  "
          f"{env_overlap(emb_obj, mf_mu, Sao):>12.1e}")

emb_obj.mu_val = mu_val

       mu    E error / Ha   max overlap


Overwritten attributes  get_hcore  of <class 'pyscf.dft.roks.ROKS'>


converged SCF energy = 25.1936110842726
    1e+03      -4.841e-05       8.2e-05
converged SCF energy = 25.1936590084292
    1e+05      -4.832e-07       8.2e-07
converged SCF energy = 25.1936594861958
    1e+07      -5.092e-09       8.2e-09
converged SCF energy = 25.1936594637705
    1e+09      -4.189e-08       8.2e-11


#### Huzinaga has no free parameter

The Huzinaga projector is $P = \gamma_B S$. Writing $\gamma_B = C_B C_B^\dagger$ ($B$ indicates the **occupied** column idxs of environment) and using
$C_B^\dagger S C_B = I$ (ortho basis), we see that it is idempotent,

$$P^2 = (C_B C_B^\dagger S) (C_B C_B^\dagger S) = C_B \; \underbrace{C_B^\dagger S C_B}_{=I} \; C_B^\dagger S = C_B C_B^\dagger S= P,$$

but **not symmetric**, so $FP$ (where $F$ is the AO Fock matrix) cannot be added to the core Hamiltonian as it stands. The operator that
can is its symmetrised form

$$O_\text{huz} = -\left(F P + P^\dagger F\right).$$

Acting on an environment orbital this returns $-\varepsilon_\text{env}$, so the occupied
environment block is reflected above the fragment's HOMO with no arbitrary constant: the shift
is set by the environment orbital energies themselves. The factor is $1$, not $1/2$ — at
$\lambda = 1$ the fragment–environment coupling block vanishes exactly, whereas $\lambda = 1/2$
zeroes only the environment block and leaves half the coupling behind.

Two implementation details that are easy to get wrong, both handled inside `build_emb_dft`:

- $O_\text{huz}$ depends on the running Fock matrix, so it cannot live in `get_hcore`, which
  PySCF evaluates *once* before the SCF loop. It goes in an overridden `get_fock` and is rebuilt
  every cycle. `get_hcore` keeps only the density-independent part, which also keeps
  `energy_elec` free of the projector.
- The SCF starts from $\gamma_A$, not PySCF's `minao` guess. Huzinaga lifts the environment by
  only $|\varepsilon_\text{env}|$, so from a poor guess the environment block can come out
  positive; the sign flip then drives those orbitals *below* the fragment's and aufbau locks
  onto a wrong but perfectly stable state.

#### 5.1.1 Note: what the Huzinaga approach actually does

Split any $S$-orthonormal set of orbitals into the occupied environment $C_B$ and everything else
$C_A$. Because $P C_B = C_B$ and $P C_A = 0$, the three blocks of $\tilde F = F + O_\text{huz}$
come out as

$$\tilde F_{AA} = F_{AA}, \qquad \tilde F_{AB} = F_{AB} - F_{AB} = 0, \qquad \tilde F_{BB} = F_{BB} - 2F_{BB} = -F_{BB}.$$

The last block is where the *twice* comes from: $FP$ and $P^\dagger F$ each contribute a full
$F_{BB}$, so the operator adds $-2F_{BB}$, and an environment level sitting at $-a$ lands at
$-a + 2a = +a$. It is pushed up by its own magnitude, with no $\mu$ to choose. This is a statement
about the *eigenvalues* of $F_{BB}$, so it holds for localised orbitals too, where $F_{BB}$ is not
diagonal.

Two things not to conflate. It is the **environment** block that gets negated... Whereas $\tilde F_{AA}$ is
left untouched, which is why the fragment's own physics is unmodified. And the reason Huzinaga is
*exact* is $\tilde F_{AB} = 0$, not the fact that the shift is self-sizing: the $\mu$-shift's AB
block is zero as well, so it leaves $F_{AB}$ intact and merely suppresses mixing as $F_{AB}/\mu$.
That is also why adding `huz_level_shift` on top cannot reintroduce an $\mathcal{O}(1/\mu)$ error
— there is no residual coupling left for a constant to fail to suppress.

**The failure mode.** If an environment eigenvalue is already *positive*, the flip moves it
**down**, making it more favourable to fill. This is live at `spin = 2`: the occupied environment
block has occupancies `[2 2 2 2 2 1 1]` and one eigenvalue of $F_{BB}$ is $+0.0363$, which lands
at $-0.0363$. The Roothaan effective Fock averages $F^\alpha$ and $F^\beta$ on its diagonal
blocks, and for a singly occupied orbital the beta partner is empty, so that average can come out
positive. Two refinements on the diagnosis: it is the eigenvalues of $F_{BB}$ that matter rather
than individual matrix elements, and the condition is not "is it positive" but "is
$-\varepsilon_\text{max}$ above the active HOMO". Here $-0.0363 > -0.4697$, so it survived with a
margin of $0.43$ Ha — the mechanism fired without breaking anything. Note there should be a warning thrown
for the user if this happens (which can be turned off with the `warn` kwarg). the `2-huz_shift.ipynb` gives
an example of scenario!

`huz_level_shift` removes the risk, because $\lambda\, S \gamma_B S$ contributes exactly
$\lambda I$ to the BB block and zero everywhere else, lifting the environment regardless of sign.
The energy is unchanged: the AA block, the vanishing coupling and the converged fragment density
are all untouched, and `emb_corr` is built from `get_hcore() - hcore_std`, which never sees the
projector. Stability insurance that costs nothing, until $\lambda$ grows large enough to eat
floating-point headroom in the eigensolver.

In [9]:
runs = {}
for label, kwargs in (
    ("mu",        dict(proj_type="mu")),
    ("huz",       dict(proj_type="huz")),
    ("huz+shift", dict(proj_type="huz", huz_level_shift=1e6)),
):
    e_tot, mf, emb_corr, env_cols, env_plus_corrections = emb_obj.build_emb_dft(xc, **kwargs)
    runs[label] = dict(e_tot=e_tot, mf=mf, env_cols=env_cols,
                       env_plus_corrections=env_plus_corrections)
    print(f"{label:>10}: E = {e_tot:.10f}   error = {e_tot - global_scf.e_tot:+.3e}   "
          f"max overlap = {env_overlap(emb_obj, mf, Sao):.1e}")

## the exactness claim, stated as a test
assert abs(runs["huz"]["e_tot"] - global_scf.e_tot) < 1e-8, "huzinaga DFT-in-DFT is not exact"

converged SCF energy = 25.1936594637705
        mu: E = -115.4037195838   error = -4.189e-08   max overlap = 8.2e-11
converged SCF energy = 25.1936594915625


Overwritten attributes  get_fock get_hcore  of <class 'pyscf.dft.roks.ROKS'>


       huz: E = -115.4037195419   error = -3.837e-11   max overlap = 9.8e-16
converged SCF energy = 25.1936594915625
 huz+shift: E = -115.4037195419   error = -3.836e-11   max overlap = 1.9e-16


### 5.2 The diagnostic, and why it must not use column indices

`emb_obj.act_cols` is a positional window into the *pre*-embedding ordering. The embedded SCF
returns its own orbitals sorted by energy, so a column index carries no meaning afterwards. The
$\mu$-shift lets you get away with it, because the environment is parked at $+\mu$ and is
therefore always in the last columns. Huzinaga shifts each environment orbital by only
$|\varepsilon_\text{env}|$, so they land scattered among the active virtuals, and a fixed index
window then picks one up — which looks like a broken projector but is not.

`check_embedding` selects orbitals by *what they are*: the weight of each converged orbital on
the occupied-environment space,

$$w_i = \left[ C^\dagger S\, \gamma_B\, S\, C \right]_{ii},$$

and calls anything above $0.5$ an environment orbital. It then asserts the two conditions the
method actually requires — the occupied orbitals span none of the environment, and no
environment orbital sits below the fragment HOMO.

In [10]:
for label, run in runs.items():
    mf = run["mf"]
    run["env_cols"] = emb_obj.check_embedding(mf.mo_coeff, mf.mo_occ, mf.mo_energy, Sao, label)

--- mu ---
  environment landed in columns : [19 20 21 22 23 24 25]
  eps(environment)              : [9.99999981e+08 9.99999990e+08 9.99999999e+08 1.00000000e+09 1.00000000e+09 1.00000000e+09 1.00000000e+09]
  occupied columns              : [0 1 2]
  eps(occupied)                 : [-1.11   -0.6157 -0.4697]
  max |<env occ| S |emb occ>|   : 8.16e-11   <- must be ~0
  aufbau margin                 : +999999981.2002 Ha  <- must be > 0
--- huz ---
  environment landed in columns : [ 3  8  9 10 14 24 25]
  eps(environment)              : [-0.0363  0.3239  0.4464  0.4951  0.6884 10.2562 19.2696]
  occupied columns              : [0 1 2]
  eps(occupied)                 : [-1.11   -0.6157 -0.4697]
  max |<env occ| S |emb occ>|   : 9.78e-16   <- must be ~0
  aufbau margin                 : +0.4334 Ha  <- must be > 0
--- huz+shift ---
  environment landed in columns : [19 20 21 22 23 24 25]
  eps(environment)              : [ 999999.9637 1000000.3239 1000000.4464 1000000.4951 1000000.6884 100

Read the `environment landed in columns` lines against each other. The $\mu$ runs put the
environment in a contiguous block at the end; plain `huz` scatters it through the active
virtuals, sometimes *below* the lowest active virtual. `huz+shift` adds a constant on top of the
Huzinaga term, which relocates the already-decoupled environment block back to the end without
reintroducing the $\mu$-shift's $1/\mu$ error — the energy is unchanged to machine precision, as
the table in 5.1 shows. That is worth having for open-shell systems: the Roothaan effective Fock
averages $F^\alpha$ and $F^\beta$, so a singly occupied environment orbital can have a positive
eigenvalue, and the sign flip then moves it *down* rather than up.

In [11]:
print(f"{'run':>10}  {'env columns':<34} {'aufbau margin / Ha':>19}")
for label, run in runs.items():
    mf = run["mf"]
    margin = (mf.mo_energy[run["env_cols"]].min()
              - mf.mo_energy[mf.mo_occ > 0].max())
    print(f"{label:>10}  {str(run['env_cols']):<34} {margin:>19.4f}")

       run  env columns                         aufbau margin / Ha
        mu  [19 20 21 22 23 24 25]                  999999981.2002
       huz  [ 3  8  9 10 14 24 25]                          0.4334
 huz+shift  [19 20 21 22 23 24 25]                    1000000.4334


### 5.3 The projector spans the occupied environment, and nothing else

$P = \gamma_B S$ must return 1 on every occupied environment orbital and 0 on everything else —
fragment orbitals *and* environment virtuals. If environment virtuals were included, the
fragment would have nowhere to relax into and the embedding would stop being exact.

In [12]:
proj_diag = np.diag(emb_obj.C_full_reidx.T @ Sao @ emb_obj.get_huz_projector()
                    @ emb_obj.C_full_reidx)
other_cols = np.setdiff1d(np.arange(mol.nao), emb_obj.env_idx_occ)

print(f"on occupied environment columns: {np.round(proj_diag[emb_obj.env_idx_occ], 10)}")
print(f"largest value anywhere else    : {np.abs(proj_diag[other_cols]).max():.2e}")

assert np.allclose(proj_diag[emb_obj.env_idx_occ], 1)
assert np.allclose(proj_diag[other_cols], 0)

on occupied environment columns: [1. 1. 1. 1. 1. 1. 1.]
largest value anywhere else    : 1.15e-16


## 6. HF-in-DFT

The machinery does not care what solves the fragment. Swapping DFT for Hartree-Fock gives a
genuine WF-in-DFT energy: exchange is now exact inside the fragment while the environment stays
at the DFT level. This one is *not* expected to reproduce the global DFT energy — the fragment
is described differently now, which is the entire point.

In [13]:
e_hf_in_dft, emb_hf, emb_corr_hf, env_cols_hf, env_plus_corr_hf = emb_obj.build_emb_hf(
    proj_type="huz"
)

emb_obj.check_embedding(emb_hf.mo_coeff, emb_hf.mo_occ, emb_hf.mo_energy, Sao, "HF-in-DFT")
print(f"\nHF-in-DFT        = {e_hf_in_dft:.10f}")
print(f"global DFT       = {global_scf.e_tot:.10f}")
print(f"whole-molecule HF= {mf_hf.e_tot:.10f}")
print(f"subsystem nelec  = {emb_hf.mol.nelec}")

converged SCF energy = 25.4609371379306
--- HF-in-DFT ---
  environment landed in columns : [ 3  4  5  6 11 24 25]
  eps(environment)              : [-0.2012 -0.0233  0.193   0.234   0.3924  6.7963 18.0875]
  occupied columns              : [0 1 2]
  eps(occupied)                 : [-1.384  -0.799  -0.6688]
  max |<env occ| S |emb occ>|   : 1.21e-15   <- must be ~0
  aufbau margin                 : +0.4676 Ha  <- must be > 0

HF-in-DFT        = -115.1364418956
global DFT       = -115.4037195419
whole-molecule HF= -114.7209283056
subsystem nelec  = (3, 3)


Overwritten attributes  get_hcore get_fock  of <class 'pyscf.scf.rohf.ROHF'>


## 7. CASCI-in-DFT, and the indexing trap

Now the payoff: a correlated solver on the fragment. This is also where the warning from the top
of the notebook bites hardest.

`mcscf.CASCI` takes its active space as a *contiguous window* around the frontier,
`range(ncore, ncore + ncas)`. After Huzinaga embedding the environment orbitals are interleaved
with the active virtuals, so that window can quietly contain environment orbitals — orbitals
whose energies are artefacts of the projector and which belong to the *other* subsystem. The CAS
would then correlate the environment.

The cell below prints the naive window alongside `env_cols_hf` so you can see the collision, then
builds the CAS from non-environment columns only. With `spin = 2` the naive window typically
swallows several environment orbitals; with `spin = 0` it often happens to miss them, which is
exactly what makes this bug so dangerous.

The occupied side needs no filtering: the environment sits above the fragment HOMO by
construction, so every occupied column is a fragment orbital.

In [14]:
n_cas_occ, n_cas_vir = 2, 4

occ_cols = np.where(emb_hf.mo_occ > 0)[0]
vir_cols_all = np.where(emb_hf.mo_occ == 0)[0]
vir_cols_safe = np.setdiff1d(vir_cols_all, env_cols_hf)

## what CASCI would pick on its own
ncore_naive = (emb_hf.mol.nelectron - 2 * n_cas_occ) // 2
naive_window = np.arange(ncore_naive, ncore_naive + n_cas_occ + n_cas_vir)
collision = np.intersect1d(naive_window, env_cols_hf)

print(f"environment columns      : {env_cols_hf}")
print(f"naive contiguous CAS     : {naive_window}")
print(f"  -> environment inside  : {collision}  "
      f"{'<-- WRONG' if len(collision) else '(none, this time)'}")

## select by role instead of position
cas_occ = occ_cols[-n_cas_occ:]
cas_vir = vir_cols_safe[:n_cas_vir]
mo_cas_idxs = np.concatenate([cas_occ, cas_vir])
ncas = len(mo_cas_idxs)

occ_in_cas = emb_hf.mo_occ[cas_occ]
nelecas = (int((occ_in_cas > 0).sum()), int((occ_in_cas > 1).sum()))

print(f"\nsafe CAS columns         : {mo_cas_idxs}")
print(f"  ncas = {ncas}   nelecas = {nelecas}")
assert not np.intersect1d(mo_cas_idxs, env_cols_hf).size, "CAS contains an environment orbital"

environment columns      : [ 3  4  5  6 11 24 25]
naive contiguous CAS     : [1 2 3 4 5 6]
  -> environment inside  : [3 4 5 6]  <-- WRONG

safe CAS columns         : [ 1  2  7  8  9 10]
  ncas = 6   nelecas = (2, 2)


In [15]:
mycas = mcscf.CASCI(emb_hf, ncas, nelecas)
mo_sorted = mcscf.addons.sort_mo(mycas, emb_hf.mo_coeff, mo_cas_idxs, base=0)
mycas.kernel(mo_sorted)

e_casci_in_dft = mycas.e_tot + env_plus_corr_hf
print(f"\nCASCI-in-DFT     = {e_casci_in_dft:.10f}")
print(f"HF-in-DFT        = {e_hf_in_dft:.10f}   "
      f"(correlation recovered {e_casci_in_dft - e_hf_in_dft:+.6f})")
print(f"whole-molecule CCSD = {cc_full.e_tot:.10f}")

CASCI E = 25.4573522691803  E(CI) = -6.83187160283465  S^2 = 0.0000000

CASCI-in-DFT     = -115.1400267643
HF-in-DFT        = -115.1364418956   (correlation recovered -0.003585)
whole-molecule CCSD = -114.9379881253


## 8. The fragment as a qubit Hamiltonian

`get_mo_integrals` returns the CAS integrals in the embedded MO basis, with `get_hcore`
overridden so that $v_\text{emb}$ is included in the one-electron part. Everything outside the
CAS is folded into `ecore`, so the constant to carry through is

$$E_\text{shift} = \texttt{env\_plus\_corrections} + \texttt{ecore}.$$

Exact diagonalisation of the qubit Hamiltonian must then reproduce the CASCI energy. Diagonalizing the Pauli operator
does not conserve particle number by itself, so penalty terms
$(\hat N_\alpha - n_\alpha)^2 + (\hat N_\beta - n_\beta)^2$ pin the ground state to the right
sector.

In [16]:
from nbed.hamiltonian import (build_molecular_H, build_number_operator,
                              build_spin_integrals)

ecore, h1e, eri = emb_obj.get_mo_integrals(
    emb_hf, emb_hf.mo_coeff, ncas, nelecas, mo_cas_idxs=mo_cas_idxs
)

h1e_spin, eri_spin = build_spin_integrals(h1e, eri, ncas)
e_shift = env_plus_corr_hf + ecore
nqubits = 2 * ncas

Hq = build_molecular_H(e_shift, h1e_spin, eri_spin)
Na, Nb = build_number_operator(nqubits, type="qubit_jw")
print(f"{nqubits} qubits, {len(Hq.terms)} Pauli terms")

12 qubits, 1815 Pauli terms


In [17]:
from openfermion import get_sparse_operator
from scipy.sparse.linalg import eigsh

assert nqubits <= 12, f"{nqubits} qubits is too many for dense diagonalisation"

H_sym = Hq + (Na - nelecas[0]) ** 2 + (Nb - nelecas[1]) ** 2
eigvals = eigsh(get_sparse_operator(H_sym).real, k=3, which="SA")[0]

print(f"qubit ground state = {eigvals[0]:.10f}")
print(f"CASCI-in-DFT       = {e_casci_in_dft:.10f}")
print(f"difference         = {eigvals[0] - e_casci_in_dft:+.2e}")
assert abs(eigvals[0] - e_casci_in_dft) < 1e-9

qubit ground state = -115.1400267643
CASCI-in-DFT       = -115.1400267643
difference         = +7.11e-14


## Summary

What was verified numerically, in order:

1. `E_act + E_env + E_cross` reproduces the global DFT electronic energy — the partition is an
   identity, not an approximation.
2. The $\mu$-shift error and the residual environment overlap both fall as $1/\mu$; Huzinaga is
   exact with no free parameter.
3. DFT-in-DFT with Huzinaga reproduces the global DFT energy to $\sim 10^{-13}$ Ha, with
   environment overlap at machine precision.
4. $P = \gamma_B S$ gives exactly 1 on occupied environment orbitals and 0 on everything else,
   including environment virtuals.
5. The naive contiguous CAS window collides with environment orbitals; selecting by projection
   weight does not.
6. The qubit Hamiltonian reproduces CASCI-in-DFT to $\sim 10^{-13}$ Ha.

Rules worth carrying into new code:

- After embedding, **select orbitals by projection weight, never by column index.** Use the
  `env_cols` returned by `build_emb_dft` / `build_emb_hf`.
- Contract $v_\text{emb}$ with the DFT fragment density $\gamma_A$, not the solver's density.
- Localise within occupation blocks only, and assert the density is unchanged.
- Prefer Huzinaga. If you need the environment parked out of the way — for open-shell aufbau
  headroom, or so that orbital windows behave — add `huz_level_shift` rather than falling back
  to the $\mu$-shift; it costs nothing in accuracy.
- Re-run this notebook with `spin = 0` and `spin = 2`. Several of the traps above are visible in
  only one of the two.